# Hong Kong Property Transactions

This notebook uses the Hong Kong transaction-level dataset as a Kaggle-style house price dataset. It also checks how much richer the data can become by joining external lookup tables and other government sources.

Note: the Kaggle House Prices competition dataset has 81 columns in the train set, including the target `SalePrice`.

## 1. Load the Kaggle-style Dataset

We start with the transaction-level Hong Kong housing dataset in this workspace. It is a good proxy for a Kaggle house-pricing dataset because each row is a property sale and the target is the transaction price.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = Path("hong_kong_housing_transactions_recent.csv")
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("Columns:", len(df.columns))
display(df.head())

Shape: (61047, 28)
Columns: 28


,source_label,scheme_type,scheme_name,asp_date,sale_year,sale_month,court_estate_name,lookup_estate_name,district_name,region_name,...,floor_num_num,flat_num,flat_num_num,saleable_area,sale_price,price_per_sq_m,court_saleable_area_low,court_saleable_area_high,court_initial_sale_price_low,court_initial_sale_price_high
0,HOS 2018,HOS,Sale of Home Ownership Scheme Flats 2018,2019-02-28,2019,2,NaN,NaN,NaN,NaN,...,23,13,13.0,44.3,3427700.0,77374.717833,NaN,NaN,NaN,NaN
1,HOS 2018,HOS,Sale of Home Ownership Scheme Flats 2018,2019-02-28,2019,2,NaN,NaN,NaN,NaN,...,38,13,13.0,44.3,3547700.0,80083.521445,NaN,NaN,NaN,NaN
2,HOS 2018,HOS,Sale of Home Ownership Scheme Flats 2018,2019-02-28,2019,2,NaN,NaN,NaN,NaN,...,38,17,17.0,44.3,3614000.0,81580.135440,NaN,NaN,NaN,NaN
3,HOS 2018,HOS,Sale of Home Ownership Scheme Flats 2018,2019-02-28,2019,2,NaN,NaN,NaN,NaN,...,39,17,17.0,44.3,3622000.0,81760.722348,NaN,NaN,NaN,NaN
4,HOS 2018,HOS,Sale of Home Ownership Scheme Flats 2018,2019-02-28,2019,2,NaN,NaN,NaN,NaN,...,35,18,18.0,44.3,3590000.0,81038.374718,NaN,NaN,NaN,NaN


## 2. Inspect Columns and Shape

The exact number of columns depends on the dataset. For this workspace dataset, we inspect the shape, column names, and a quick sample.

In [ ]:
print("Shape:", df.shape)
print("Column count:", df.shape[1])
print("Columns:")
for col in df.columns:
    print("-", col)

display(df.sample(5, random_state=42))

## 3. Check Missing Values and Duplicates

Before adding more data, it helps to check data quality.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
duplicates = df.duplicated().sum()

print("Missing values per column:")
print(missing[missing > 0])
print("\nDuplicate rows:", duplicates)

## 4. Gather Additional Data

We can enrich the transaction records with court-level metadata from the Hong Kong Housing Authority API, including district, region, year of completion, and building type.

In [ ]:
from io import BytesIO
from urllib.request import Request, urlopen
import json
import re

COURTS_URL = "https://data.housingauthority.gov.hk/psi/rest/export/hos-courts"


def normalize_name(value: str) -> str:
    return re.sub(r"[^A-Z0-9]+", "", str(value).upper())


request = Request(COURTS_URL, headers={"User-Agent": "Mozilla/5.0"})
with urlopen(request, timeout=90) as response:
    courts_payload = json.loads(response.read())

courts = pd.DataFrame(courts_payload["data"])
courts["join_key"] = courts["estate_name"].map(normalize_name)

courts = courts.rename(
    columns={
        "estate_name": "lookup_estate_name",
        "district_name": "district_name",
        "region_name": "region_name",
        "year_of_completion": "year_of_completion",
        "type_of_block": "type_of_block",
        "no_of_blocks": "no_of_blocks",
        "no_of_flats": "no_of_flats",
    }
)

courts["year_of_completion"] = pd.to_numeric(courts["year_of_completion"], errors="coerce")
print("Court lookup rows:", len(courts))
display(courts[["lookup_estate_name", "district_name", "region_name", "year_of_completion", "type_of_block"]].head())

## 5. Merge or Append New Data

We merge the lookup table onto the transactions using normalized estate names. This expands the original dataset with location and building attributes that help explain price differences.

In [ ]:
expanded = df.copy()
expanded["join_key"] = expanded["court_estate_name"].map(normalize_name)

expanded = expanded.merge(
    courts[[
        "join_key",
        "lookup_estate_name",
        "district_name",
        "region_name",
        "year_of_completion",
        "type_of_block",
        "no_of_blocks",
        "no_of_flats",
    ]],
    on="join_key",
    how="left",
)

expanded["building_age_at_sale"] = expanded["sale_year"] - expanded["year_of_completion"]
expanded.loc[expanded["building_age_at_sale"] < 0, "building_age_at_sale"] = np.nan
expanded["price_per_sq_m"] = expanded["sale_price"] / expanded["saleable_area"]

print("Expanded shape:", expanded.shape)
display(expanded.head())

## 6. Validate the Expanded Dataset

After merging, recheck the final shape, the new columns, missing values, and a few sample rows.

In [ ]:
print("Expanded shape:", expanded.shape)
print("Expanded columns:", len(expanded.columns))

expanded_missing = expanded.isna().sum().sort_values(ascending=False)
print("Missing values after merge:")
print(expanded_missing[expanded_missing > 0])

print("Duplicate rows after merge:", expanded.duplicated().sum())
display(expanded.sample(5, random_state=7))

In [1]:
import polars as pl

In [4]:
df = pl.read_csv(
    "data_outputs/hk_apartment_transactions_enriched.csv",
    schema_overrides={"flat_num": pl.String},
)

In [6]:
len(df.columns)

37